# OUR 음수값 클리핑 민감도 분석

산소소모율(OUR)의 음수값을 0으로 바꿨을 때 배치 요약값과 성과지표의 상관이 얼마나 달라지는지 확인한다. 원본 데이터는 수정하지 않고 메모리에서만 비교하며, CSV는 저장하지 않는다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
print(f'{data.shape[0]:,}행, {data["배치번호"].nunique()}개 배치')

### 확인

입력은 113,935행, 100개 배치다. 이후 계산되는 `OUR_0클리핑`은 비교용 임시 변수이며 원본 열을 덮어쓰지 않는다.

In [ ]:
rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    penicillin = batch['페니실린농도(g/L)'].to_numpy()
    our_raw = batch['산소소모율(g/min)'].to_numpy()
    our_clipped = np.clip(our_raw, 0, None)
    late = time / time[-1] >= 0.8
    rows.append({
        '배치번호': batch_number,
        '그룹': '정상' if batch_number <= 90 else 'Fault',
        'OUR원본평균': our_raw.mean(),
        'OUR클리핑평균': our_clipped.mean(),
        '평균차이': our_clipped.mean() - our_raw.mean(),
        'OUR음수비율': np.mean(our_raw < 0),
        '최종농도': penicillin[-1],
        '총수확량': batch['총수확량(kg)'].iloc[0],
        '농도유지율': penicillin[-1] / penicillin.max(),
        '생산성': penicillin[-1] / time[-1],
        '후기농도기울기': np.polyfit(time[late], penicillin[late], 1)[0],
    })
batch_metrics = pd.DataFrame(rows)
summary = batch_metrics[['OUR원본평균', 'OUR클리핑평균', '평균차이']].agg(['mean', 'median', 'max'])
display(summary.round(8))
print('클리핑 영향 배치:', int((batch_metrics['평균차이'] > 0).sum()), '/ 100')

### 판단

100개 배치 모두에 음수 OUR이 조금씩 포함되어 있다. 클리핑 후 배치 평균은 전체 평균 기준 1.262653에서 1.263307로 0.000654 증가했고, 가장 큰 배치 변화도 0.002755에 불과하다. 방향은 일관되지만 절대 변화량은 매우 작다.

In [ ]:
paired_t = stats.ttest_rel(batch_metrics['OUR클리핑평균'], batch_metrics['OUR원본평균'])
wilcoxon = stats.wilcoxon(batch_metrics['OUR클리핑평균'], batch_metrics['OUR원본평균'])
print(f'대응 t-test: t={paired_t.statistic:.6f}, p={paired_t.pvalue:.3e}')
print(f'Wilcoxon: W={wilcoxon.statistic:.6f}, p={wilcoxon.pvalue:.3e}')

normal = batch_metrics.loc[batch_metrics['배치번호'] <= 90]
outcomes = ['최종농도', '총수확량', '농도유지율', '생산성', '후기농도기울기']
correlation_rows = []
for outcome in outcomes:
    raw = stats.spearmanr(normal['OUR원본평균'], normal[outcome])
    clipped = stats.spearmanr(normal['OUR클리핑평균'], normal[outcome])
    correlation_rows.append({
        '성과': outcome, '원본_rho': raw.statistic, '클리핑_rho': clipped.statistic,
        'rho차이': clipped.statistic - raw.statistic,
    })
display(pd.DataFrame(correlation_rows).round(6))
display(batch_metrics.groupby('그룹')['OUR음수비율'].agg(['mean', 'median', 'max']).round(6))

### 최종 판단

- 대응 t-test와 Wilcoxon은 모두 유의하지만, 이는 100개 배치에서 같은 방향의 아주 작은 변화가 누적된 결과다. 통계적 유의성과 실무적 중요성은 구분해야 한다.
- 원본과 클리핑 OUR의 성과 상관 차이는 모든 지표에서 |Δρ|<0.0025다. 가장 큰 변화도 최종농도의 -0.00234이므로 현재 배치 수준 결론은 클리핑 여부에 사실상 민감하지 않다.
- Fault의 OUR 음수비율 평균은 0.0141로 정상의 0.0086보다 높다. 음수값 자체가 센서 잡음 또는 공정 이상 신호일 수 있으므로 일괄 삭제하지 않는다.
- 권장안은 원본 OUR을 보존하고 `OUR음수비율`을 품질 플래그로 사용하는 것이다. 0 클리핑 값은 물리적 해석이나 민감도 확인용 파생변수로만 둔다.